### Objective
Create a machine learning model to predict if a bank client is about to cancel the account.

In [1]:
import pandas as pd
import sklearn
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OrdinalEncoder, StandardScaler
from sklearn.metrics import f1_score, roc_auc_score
from sklearn.utils import shuffle
import plotly.express as px

import numpy as np

In [2]:
print(f'pandas version: {pd.__version__}')
print(f'scikit-learn version: {sklearn.__version__}')

pandas version: 3.0.5
scikit-learn version: 1.9.0


### Functions

In [3]:
def lr_model_scores(features_train, target_train, features_val, target_val,
                    weight=None):
    '''
    Returns LogisticRegression model end its scores.
    '''
    model = LogisticRegression(
        solver='liblinear',
        random_state=123,
        class_weight=weight
    )
    model.fit(features_train, target_train)

    predicted_valid = model.predict(features_val)
    f1 = f1_score(target_val, predicted_valid)

    proba_valid = model.predict_proba(features_val)
    proba_one_valid = proba_valid[:, 1]
    roc_auc = roc_auc_score(target_val, proba_one_valid)

    print(
        f'F1 Score: {f1:.3f}\n'
        f'ROC AUC: {roc_auc:.3f}'
    )

    return model, f1, roc_auc


def rfc_optimizer(features_train, target_train, features_val, target_val,
                  weight=None):
    '''
    Returns a RandonForestClassifier model with best n_estimators between 
    10 and 150 that are multiple of ten, and best max_depth between 5 and 15, 
    maximizing the F1 Score. Also plots the F1 score for max_depth between 1
    and 20 for the best n_estimators, to verify if there is no better option 
    out of the tested range.
    '''

    best_model = None
    best_n_estimators = 0
    best_depth = 0
    best_f1_score = 0

    for estimators in range(10, 151, 10):
        for depth in range(5, 16):
            model = RandomForestClassifier(
                n_estimators=estimators,
                max_depth=depth,
                random_state=123,
                class_weight=weight
            )
            model.fit(features_train, target_train)

            predicted_valid = model.predict(features_val)
            f1 = f1_score(target_val, predicted_valid)

            if f1 > best_f1_score:
                best_n_estimators = estimators
                best_depth = depth
                best_f1_score = f1
                best_model = model

    proba_valid = best_model.predict_proba(features_val)
    proba_one_valid = proba_valid[:, 1]
    roc_auc = roc_auc_score(target_val, proba_one_valid)

    print(
        f'best_depth: {best_depth}\n'
        f'best_n_estimators: {best_n_estimators}\n'
        f'F1 Score: {best_f1_score:.3f}\n'
        f'ROC AUC: {roc_auc:.3f}'
    )

    f1_to_plot = pd.DataFrame(columns=['depth', 'f1_val'])
    for depth in range(1, 21):
        model = RandomForestClassifier(
            n_estimators=best_n_estimators,
            max_depth=depth,
            random_state=123,
            class_weight=weight
        )

        model.fit(features_train, target_train)    

        predicted_valid = model.predict(features_val)
        f1_val = f1_score(target_val, predicted_valid)
    
        f1_to_plot.loc[len(f1_to_plot)] = [depth, f1_val]
    
    fig = px.line(
        f1_to_plot,
        x='depth',
        y='f1_val',
        title=f'Validation F1 by Depth for n_estimators = {best_n_estimators}',
        labels={'f1_val': 'F1 Score', 'depth': 'Max Depth'},
        markers=True
    )
    fig.show()

    return best_model, best_n_estimators, best_depth, best_f1_score, roc_auc


def dtc_optimizer(features_train, target_train, features_val, target_val,
                  weight=None):
    '''
    Returns a DecisionTreeClassifier model with best max_depth between 
    1 and 20, maximizing the F1 Score. Also plots the F1 score for max_depth 
    between 1 and 20 for the best n_estimators.
    '''
    f1_to_plot = pd.DataFrame(columns=['depth', 'f1_val'])

    best_model = None
    best_depth = 0
    best_f1_score = 0
    for depth in range(1, 21):
        model = DecisionTreeClassifier(
            max_depth=depth, 
            random_state=123,
            class_weight=weight
        )

        model.fit(features_train, target_train)
        
        predicted_valid = model.predict(features_val)
        f1 = f1_score(target_val, predicted_valid)

        f1_to_plot.loc[len(f1_to_plot)] = [depth, f1]
        
        if f1 > best_f1_score:
            best_depth = depth
            best_f1_score = f1
            best_model = model

    proba_valid = best_model.predict_proba(features_val)
    proba_one_valid = proba_valid[:, 1]
    roc_auc = roc_auc_score(target_val, proba_one_valid)

    print(
        f'best_depth: {best_depth}\n'
        f'F1 Score: {best_f1_score:.3f}\n'
        f'ROC AUC: {roc_auc:.3f}'
    )

    fig = px.line(
        f1_to_plot,
        x='depth',
        y='f1_val',
        title=f'Validation F1 by Depth',
        labels={'f1_val': 'F1 Score', 'depth': 'Max Depth'},
        markers=True
    )
    fig.show()
           
    return best_model, best_depth, best_f1_score, roc_auc


def upsample(features, target):
    '''upsample df by multiplying it where target is 1'''
    # number to multiply the minority class
    multiplier = round(target.value_counts()[0] / target.value_counts()[1])
    
    features_zero = features[target == 0]
    features_one = features[target == 1]
    target_zero = target[target == 0]
    target_one = target[target == 1]

    features_up = pd.concat([features_zero] + [features_one] * multiplier)
    target_up = pd.concat([target_zero] + [target_one] * multiplier)

    features_up, target_up = shuffle(features_up, target_up, random_state=123)

    return features_up, target_up


def downsample(features, target):
    '''downsample df by sampling a fraction where target is 0'''
    fraction = target.value_counts()[1] / target.value_counts()[0]

    features_zero = features[target == 0]
    features_one = features[target == 1]
    target_zero = target[target == 0]
    target_one = target[target == 1]

    features_down = pd.concat(
        [features_zero.sample(frac=fraction, random_state=123)] + 
        [features_one]
    )
    target_down = pd.concat(
        [target_zero.sample(frac=fraction, random_state=123)] +
        [target_one]
    )

    features_down, target_down = shuffle(
        features_down,
        target_down,
        random_state=123
    )

    return features_down, target_down     

### Read CSV

In [4]:
df = pd.read_csv('./datasets/Churn.csv')
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 14 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   RowNumber        10000 non-null  int64  
 1   CustomerId       10000 non-null  int64  
 2   Surname          10000 non-null  str    
 3   CreditScore      10000 non-null  int64  
 4   Geography        10000 non-null  str    
 5   Gender           10000 non-null  str    
 6   Age              10000 non-null  int64  
 7   Tenure           9091 non-null   float64
 8   Balance          10000 non-null  float64
 9   NumOfProducts    10000 non-null  int64  
 10  HasCrCard        10000 non-null  int64  
 11  IsActiveMember   10000 non-null  int64  
 12  EstimatedSalary  10000 non-null  float64
 13  Exited           10000 non-null  int64  
dtypes: float64(3), int64(8), str(3)
memory usage: 1.1 MB


In [5]:
df.head()

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2.0,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1.0,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8.0,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1.0,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2.0,125510.82,1,1,1,79084.10,0


The column 'Exited' shows if the costumer cancelled (1) or not (0). It is the model prediction objective.

## 1 - Cleaning

In [6]:
df.duplicated().sum()

np.int64(0)

In [7]:
df['CustomerId'].duplicated().sum()

np.int64(0)

There is no duplicated row or 'CustomerId'.

The columns 'RowNumber', 'CustomerId' and 'Surname' are not relevant for model training and can be discarded.

In [8]:
df = df.drop(['RowNumber', 'CustomerId', 'Surname'], axis=1)

There are null values in the 'Tenure' column. Verify if they are 0.

In [9]:
df['Tenure'].value_counts(dropna=False)

Tenure
1.0     952
2.0     950
8.0     933
3.0     928
5.0     927
7.0     925
NaN     909
4.0     885
9.0     882
6.0     881
10.0    446
0.0     382
Name: count, dtype: int64

These null are not 0, because there are observations with 0.

Verify if the rows differ from rows with values.

In [10]:
df[df['Tenure'].isna()].sample(10, random_state=123)

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
7811,611,France,Male,30,NaN,104145.65,1,0,0,159629.64,0
3543,675,France,Male,49,NaN,135133.39,1,0,1,179521.24,1
6471,850,France,Female,40,NaN,0.00,2,1,0,35034.15,0
94,730,Spain,Male,42,NaN,0.00,2,0,1,85982.47,0
6187,704,France,Male,36,NaN,114370.41,1,0,1,66810.48,0
7026,677,France,Female,27,NaN,0.00,2,0,1,114685.92,0
5314,743,Germany,Male,45,NaN,144677.19,3,1,0,22512.44,1
4445,739,France,Male,29,NaN,0.00,2,1,1,164484.78,0
9006,661,Spain,Male,20,NaN,0.00,1,1,0,110252.53,0
1960,655,Spain,Female,36,NaN,135515.76,1,1,0,86013.96,0


In [11]:
df[df['Tenure'].notna()].describe()

,CreditScore,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
count,9091.000000,9091.000000,9091.000000,9091.000000,9091.000000,9091.000000,9091.000000,9091.000000,9091.000000
mean,650.736553,38.949181,4.997690,76522.740015,1.530195,0.704983,0.515565,100181.214924,0.203938
std,96.410471,10.555581,2.894723,62329.528576,0.581003,0.456076,0.499785,57624.755647,0.402946
min,350.000000,18.000000,0.000000,0.000000,1.000000,0.000000,0.000000,11.580000,0.000000
25%,584.000000,32.000000,2.000000,0.000000,1.000000,0.000000,0.000000,51227.745000,0.000000
50%,652.000000,37.000000,5.000000,97318.250000,1.000000,1.000000,1.000000,100240.200000,0.000000
75%,717.000000,44.000000,7.000000,127561.890000,2.000000,1.000000,1.000000,149567.210000,0.000000
max,850.000000,92.000000,10.000000,250898.090000,4.000000,1.000000,1.000000,199992.480000,1.000000


In [12]:
df[df['Tenure'].isna()].describe()

,CreditScore,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
count,909.000000,909.000000,0.0,909.000000,909.000000,909.000000,909.000000,909.000000,909.000000
mean,648.451045,38.647965,NaN,76117.341474,1.530253,0.710671,0.510451,99180.389373,0.201320
std,99.079381,9.785438,NaN,63105.690715,0.588452,0.453701,0.500166,56378.063765,0.401207
min,359.000000,18.000000,NaN,0.000000,1.000000,0.000000,0.000000,106.670000,0.000000
25%,580.000000,32.000000,NaN,0.000000,1.000000,0.000000,0.000000,49872.330000,0.000000
50%,647.000000,37.000000,NaN,96674.550000,1.000000,1.000000,1.000000,99444.020000,0.000000
75%,718.000000,43.000000,NaN,128554.980000,2.000000,1.000000,1.000000,145759.700000,0.000000
max,850.000000,92.000000,NaN,206663.750000,4.000000,1.000000,1.000000,199390.450000,1.000000


In [13]:
print(df[df['Tenure'].notna()]['Gender'].value_counts(normalize=True))
print()
print(df[df['Tenure'].isna()]['Gender'].value_counts(normalize=True))


Gender
Male      0.547135
Female    0.452865
Name: proportion, dtype: float64

Gender
Male      0.531353
Female    0.468647
Name: proportion, dtype: float64


In [14]:
print(df[df['Tenure'].notna()]['Geography'].value_counts(normalize=True))
print()
print(df[df['Tenure'].isna()]['Geography'].value_counts(normalize=True))


Geography
France     0.500495
Germany    0.252227
Spain      0.247278
Name: proportion, dtype: float64

Geography
France     0.510451
Spain      0.251925
Germany    0.237624
Name: proportion, dtype: float64


In [15]:
print(df[df['Tenure'].notna()]['Exited'].value_counts(normalize=True))
print()
print(df[df['Tenure'].isna()]['Exited'].value_counts(normalize=True))


Exited
0    0.796062
1    0.203938
Name: proportion, dtype: float64

Exited
0    0.79868
1    0.20132
Name: proportion, dtype: float64


The rows with and without null values in 'Tenure' column have similar distributions, especially in the target column 'Exited'. This means that this rows can be discarded without changing the target balance.

In [16]:
df = df.dropna().reset_index(drop=True)

Verify if age makes sense.

In [17]:
df['Age'].describe()

count    9091.000000
mean       38.949181
std        10.555581
min        18.000000
25%        32.000000
50%        37.000000
75%        44.000000
max        92.000000
Name: Age, dtype: float64

No values below 18 or very high values.

## 2 - Preparing

### 2.1 - Class Imbalance Analysis

In [18]:
df['Exited'].value_counts(normalize=True)

Exited
0    0.796062
1    0.203938
Name: proportion, dtype: float64

Only 20% of the values are 1. This will not be balanced now to compare the model before and after considering the imbalance.

### 2.2 - Encode Cathegorical Characteristics

In [19]:
df['Geography'].unique()

<StringArray>
['France', 'Spain', 'Germany']
Length: 3, dtype: str

In [20]:
df['Gender'].unique()

<StringArray>
['Female', 'Male']
Length: 2, dtype: str

LogisticRegression performs better with One-Hot Encoding, 
while RandomForestClassifier and DecisionTreeClassifier perform better with Label Encoding.

Two dfs will be created, one for each encoding technique.

#### 2.2.1 One-Hot Encoding

In [21]:
df_ohe = pd.get_dummies(df, drop_first=True)
df_ohe.head()

,CreditScore,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Geography_Germany,Geography_Spain,Gender_Male
0,619,42,2.0,0.00,1,1,1,101348.88,1,False,False,False
1,608,41,1.0,83807.86,1,0,1,112542.58,0,False,True,False
2,502,42,8.0,159660.80,3,1,0,113931.57,1,False,False,False
3,699,39,1.0,0.00,2,0,0,93826.63,0,False,False,False
4,850,43,2.0,125510.82,1,1,1,79084.10,0,False,True,False


#### 2.2.2 Label Encoding

In [22]:
cat_columns = ['Geography', 'Gender']

encoder = OrdinalEncoder()
df_cat = pd.DataFrame(
    encoder.fit_transform(df[cat_columns]), 
    columns=cat_columns
)

df_le = df.copy()
df_le[cat_columns] = df_cat
df_le.head()


,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,619,0.0,0.0,42,2.0,0.00,1,1,1,101348.88,1
1,608,2.0,0.0,41,1.0,83807.86,1,0,1,112542.58,0
2,502,0.0,0.0,42,8.0,159660.80,3,1,0,113931.57,1
3,699,0.0,0.0,39,1.0,0.00,2,0,0,93826.63,0
4,850,2.0,0.0,43,2.0,125510.82,1,1,1,79084.10,0


### 2.3 - Splitting

In [23]:
# split df_ohe into train (0.6), validation (0.2) and test (0.2)
df_train, df_val_test = train_test_split(
    df_ohe,
    stratify=df_ohe['Exited'],
    test_size=0.4, 
    random_state=123
)
df_val, df_test = train_test_split(
    df_val_test,
    stratify=df_val_test['Exited'],
    test_size=0.5,
    random_state=123
)

features_train_ohe = df_train.drop('Exited', axis=1)
target_train_ohe = df_train['Exited']

features_val_ohe = df_val.drop('Exited', axis=1)
target_val_ohe = df_val['Exited']

features_test_ohe = df_test.drop('Exited', axis=1)
target_test_ohe = df_test['Exited']

In [24]:
# split df_le into train (0.6), validation (0.2) and test (0.2)
df_train, df_val_test = train_test_split(
    df_le,
    stratify=df_le['Exited'],
    test_size=0.4,
    random_state=123
)
df_val, df_test = train_test_split(
    df_val_test,
    stratify=df_val_test['Exited'],
    test_size=0.5,
    random_state=123
)
features_train_le = df_train.drop('Exited', axis=1)
target_train_le = df_train['Exited']

features_val_le = df_val.drop('Exited', axis=1)
target_val_le = df_val['Exited']

features_test_le = df_test.drop('Exited', axis=1)
target_test_le = df_test['Exited']

### 2.4 - Scaling

df_le is used only on DecisionTreeClassifier and RandomForestClassifier and do not need scaling.

In [25]:
# columns to scale
cols_sca = ['CreditScore', 'Age', 'Tenure', 'Balance', 'NumOfProducts', 
            'EstimatedSalary']

# scale df_ohe
scaler_ohe = StandardScaler()
scaler_ohe.fit(features_train_ohe[cols_sca])

features_train_ohe[cols_sca] = scaler_ohe.transform(features_train_ohe[cols_sca])
features_val_ohe[cols_sca] = scaler_ohe.transform(features_val_ohe[cols_sca])
features_test_ohe[cols_sca] = scaler_ohe.transform(features_test_ohe[cols_sca])


## 3 - Model with Imbalance
Model without correcting the target imbalance.

In [26]:
# data frame to store the results
scores_df = pd.DataFrame(columns=[
    'model',
    'n_estimators',
    'depth',
    'f1',
    'auc_roc']
)


### 3.1 - LogisticRegression
Logistic Regression model performs better with One-hot encoding.

In [27]:
model_lr, f1_lr, roc_auc_lr = lr_model_scores(
    features_train_ohe,
    target_train_ohe,
    features_val_ohe,
    target_val_ohe
)

scores_df.loc[len(scores_df)] = ['model_lr','','', f1_lr, roc_auc_lr]

F1 Score: 0.361
ROC AUC: 0.773


### 3.2 - RandomForestClassifier

Random Forest model performs better with Label Encoding.

In [28]:
model_rfc, n_estimators, depth, f1_rfc, roc_auc_rfc = rfc_optimizer(
    features_train_le,
    target_train_le,
    features_val_le,
    target_val_le
)


scores_df.loc[len(scores_df)] = [
    'model_rfc', 
    n_estimators, 
    depth, 
    f1_rfc, 
    roc_auc_rfc
]


best_depth: 15
best_n_estimators: 40
F1 Score: 0.583
ROC AUC: 0.841


### 3.3 - DecisionTreeClassifier

In [29]:
model_dtc, depth, f1_dtc, roc_auc_dtc = dtc_optimizer(
    features_train_le,
    target_train_le,
    features_val_le,
    target_val_le
)

scores_df.loc[len(scores_df)] = ['model_dtc','',depth, f1_dtc, roc_auc_dtc]

best_depth: 6
F1 Score: 0.535
ROC AUC: 0.832


The F1 score is much worse for LogisticRegression with the imbalance than for 
the other models.

The ROC-AUC is also lower for LogisticRegression and very close for the other two.

In all three cases the F1 is lower than the ROC-AUC.

## 4 - Model without Imbalance
Model correcting the target imbalance.

In [30]:
# one-hot encoding with upsample
features_train_ohe_up, target_train_ohe_up = upsample(features_train_ohe,
                                                      target_train_ohe)

# label encoding with upsample
features_train_le_up, target_train_le_up = upsample(features_train_le,
                                                    target_train_le)

# one-hot encoding with downsample
features_train_ohe_down, target_train_ohe_down = downsample(features_train_ohe,
                                                            target_train_ohe)

# label encoding with downsample
features_train_le_down, target_train_le_down = downsample(features_train_le,
                                                          target_train_le)

### 4.1 LogisticRegression


#### 4.1.1 - Model Class Weight Parameter
Correcting with model class_weight='balanced' parameter, that changes the weight to compensate the imbalance.

In [31]:
model_lr_we, f1_lr_we, roc_auc_lr_we = lr_model_scores(
    features_train_ohe,
    target_train_ohe,
    features_val_ohe,
    target_val_ohe,
    weight='balanced'
)
scores_df.loc[len(scores_df)] = ['model_lr_we','','', f1_lr_we, roc_auc_lr_we]

F1 Score: 0.493
ROC AUC: 0.777


#### 4.1.2 - Upsample

In [32]:
model_lr_up, f1_lr_up, roc_auc_lr_up = lr_model_scores(
    features_train_ohe_up,
    target_train_ohe_up,
    features_val_ohe,
    target_val_ohe,
)

scores_df.loc[len(scores_df)] = ['model_lr_up','','', f1_lr_up, roc_auc_lr_up]

F1 Score: 0.489
ROC AUC: 0.777


#### 4.1.3 - Downsample

In [33]:
model_lr_down, f1_lr_down, roc_auc_lr_down = lr_model_scores(
    features_train_ohe_down,
    target_train_ohe_down,
    features_val_ohe,
    target_val_ohe,
)

scores_df.loc[len(scores_df)] = [
    'model_lr_down',
    '',
    '',
    f1_lr_down,
    roc_auc_lr_down]

F1 Score: 0.493
ROC AUC: 0.774


The three imbalance correction techniques for Logistic Regression produce very close scores.

### 4.2 - RandomForestClassifier

#### 4.2.1 - Model Class Weight Parameter

In [34]:
model_rfc_we, n_estimators, depth, f1_rfc_we, roc_auc_rfc_we = rfc_optimizer(
    features_train_le,
    target_train_le,
    features_val_le,
    target_val_le,
    weight='balanced'
)

scores_df.loc[len(scores_df)] = [
    'model_rfc_we', 
    n_estimators, 
    depth,
    f1_rfc_we,
    roc_auc_rfc_we]

best_depth: 8
best_n_estimators: 100
F1 Score: 0.621
ROC AUC: 0.861


#### 4.2.2 - Upsample

In [35]:
model_rfc_up, n_estimators, depth, f1_rfc_up, roc_auc_rfc_up = rfc_optimizer(
    features_train_le_up,
    target_train_le_up,
    features_val_le,
    target_val_le,
)

scores_df.loc[len(scores_df)] = [
    'model_rfc_up',
    n_estimators,
    depth,
    f1_rfc_up,
    roc_auc_rfc_up
]

best_depth: 12
best_n_estimators: 60
F1 Score: 0.615
ROC AUC: 0.852


#### 4.2.3 - Downsample

In [36]:
model_rfc_down, n_estimators, depth, f1_rfc_down, roc_auc_rfc_down = (
rfc_optimizer(
    features_train_le_down,
    target_train_le_down,
    features_val_le,
    target_val_le,
))

scores_df.loc[len(scores_df)] = [
    'model_rfc_down',
    n_estimators,
    depth,
    f1_rfc_down,
    roc_auc_rfc_down
]

best_depth: 5
best_n_estimators: 100
F1 Score: 0.606
ROC AUC: 0.859


### 4.3 - DecisionTreeClassifier

#### 4.3.1 - Model Class Weight Parameter

In [37]:
model_dtc_we, depth, f1_dtc_we, roc_auc_dtc_we = dtc_optimizer(
    features_train_le,
    target_train_le,
    features_val_le,
    target_val_le,
    weight='balanced'
)

scores_df.loc[len(scores_df)] = [
    'model_dtc_we',
    '',
    depth,
    f1_dtc_we,
    roc_auc_dtc_we
]

best_depth: 5
F1 Score: 0.572
ROC AUC: 0.828


#### 4.3.2 - Upsample

In [38]:
model_dtc_up, depth, f1_dtc_up, roc_auc_dtc_up = dtc_optimizer(
    features_train_le_up,
    target_train_le_up,
    features_val_le,
    target_val_le
)

scores_df.loc[len(scores_df)] = [
    'model_dtc_up',
    '',
    depth,
    f1_dtc_up,
    roc_auc_dtc_up]

best_depth: 5
F1 Score: 0.572
ROC AUC: 0.827


#### 4.3.3 - Downsample

In [39]:
model_dtc_down, depth, f1_dtc_down, roc_auc_dtc_down = dtc_optimizer(
    features_train_le_down,
    target_train_le_down,
    features_val_le,
    target_val_le
)

scores_df.loc[len(scores_df)] = [
    'model_dtc_down',
    '',
    depth,
    f1_dtc_down,
    roc_auc_dtc_down
]

best_depth: 6
F1 Score: 0.573
ROC AUC: 0.816


The three imbalance correction techniques for Decision Tree Classifier produce very close scores.

## 5 - F1 and AUC ROC Scores

The three imbalance correction techniques for Logistic Regression produce very close scores.scores_df.round(3)

The F1 score increases for all models when using balanced data, but the ROC-AUC remains almost the same. 

This happens because balancing change the scale of the probabilities, increasing the F1 score at threshold=0.5. ROC-AUC measures model the ranking capacity, that does not change with balancing.

The best model obtained is the Random Forest Classifier trained with class weight parameter.

## 6 - Testing

Retrain the model with validation and training data together before the final test.

In [41]:
features_train_val_le = pd.concat(
    [features_train_le, features_val_le], ignore_index=True
)
target_train_val_le = pd.concat(
    [target_train_le, target_val_le], ignore_index=True
)

model_rfc_we.fit(features_train_val_le, target_train_val_le)

,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",8
,"random_state random_state: int, RandomState instance or None, default=NoneControls both the randomness of the bootstrapping of the samples usedwhen building trees (if ``bootstrap=True``) and the sampling of thefeatures to consider when looking for the best split at each node(if ``max_features < n_features``).See :term:`Glossary <random_state>` for details.",123
,"class_weight class_weight: {""balanced"", ""balanced_subsample""}, dict or list of dicts, default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one. Formulti-output problems, a list of dicts can be provided in the sameorder as the columns of y.Note that for multioutput (including multilabel) weights should bedefined for each class of every column in its own dict. For example,for four-class multilabel classification weights should be[{0: 1, 1: 1}, {0: 1, 1: 5}, {0: 1, 1: 1}, {0: 1, 1: 1}] instead of[{1:1}, {2:5}, {3:1}, {4:1}].The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``The ""balanced_subsample"" mode is the same as ""balanced"" except thatweights are computed based on the bootstrap sample for every treegrown.For multi-output, the weights of each column of y will be multiplied.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified.",'balanced'
,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",100
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 

In [42]:

predicted_test = model_rfc_we.predict(features_test_le)
f1 = f1_score(target_test_le, predicted_test)

proba_test = model_rfc_we.predict_proba(features_test_le)
proba_one_test = proba_test[:, 1]
roc_auc = roc_auc_score(target_test_le, proba_one_test)

print(
    f'Test F1 Score: {f1:.3f}\n'
    f'Test ROC AUC: {roc_auc:.3f}'
)


Test F1 Score: 0.583
Test ROC AUC: 0.845
